# Averaged (across problems ≥ cutoff) good-reversal choice-probability plots — **fixed cumulative axis**

This notebook reproduces your script’s plot variants, **averaged across problems ≥ `CUTOFF_PROBLEM`**, and fixes two issues:

1. **Cumulative axis fraction removed is correct**  
   We compute `fraction_removed` **within each problem** (never by merging windows across problems), then average the fraction-removed curves across problems.

2. **Titles show real counts**  
   For each averaged plot:
   - `n_subjects` = union of subjects that contributed ≥1 good reversal in the included problems (or split)
   - `n_reversals` = total number of good reversals included (sum across problems)

We still use your existing `plot_choice_probs_around_good_reversals` for plots **without** cumulative fraction removed.
For plots **with** the fraction-removed axis, we use a small wrapper that draws the same lines + a twin axis.

In [1]:
# --- Imports / path setup (match your script) ---
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))

import src.behavior_visualization.plot_style  # noqa: F401 — applies rcParams

from src.behavior_import.import_data import *
from src.behavior_import.extract_trials import *
from src.behavior_analysis.get_total_reversals import *
from src.behavior_analysis.get_good_reversal_info import *
from src.behavior_analysis.get_choice_probs_around_good_reversals import *
from src.behavior_analysis.split_good_reversals_by_best_change import *
from src.behavior_analysis.split_early_late_good_reversals import *
from src.behavior_visualization.plot_choice_probs_around_good_reversals import *
from fix_grid_maze_cohort_02_problems import *

## Parameters (mirror your script)

In [2]:
task = "grid-maze"
# task = "open-field"

# which plot families to generate
DO_ALL = True
DO_SKIP = True
DO_MOVING_AVG = True
DO_REMOVE_BAD = True
DO_SPLIT_BY_BEST_CHANGE = False
DO_SPLIT_BY_BEST_CHANGE_AND_HALF = False
DO_SPLIT_BY_FIRST_TWO = True
DO_SPLIT_BY_HALF = True

# averaging across problems
CUTOFF_PROBLEM = 7   # average problems >= this

# window params
pre = 10
post = 30
skip_n_trials_after_reversal = 15

# smoothing
moving_avg_window = 4

# output base
OUT_BASE = Path("../results/figures")  # same as your script


## Load data and group into problems

In [3]:
folder_name = None
cohort = None
if task == "grid-maze":
    cohort = "cohort-02"
    folder_name = "3x3_maze_blocked_reward_bandit"
elif task == "open-field":
    cohort = "cohort-01"
    folder_name = "3x3_field_blocked_reward_bandit"

root = f"/Volumes/behrens/meg/{folder_name}/{cohort}/rawdata/"

subjects_data = import_data(root)
subjects_trials_by_problem = extract_trials_grouped_by_problem(subjects_data)

if task == "grid-maze" and cohort == "cohort-02":
    subjects_trials_by_problem = fix_grid_maze_cohort_02_problems(subjects_trials_by_problem)

problem_numbers = sorted(subjects_trials_by_problem.keys())
print("Problems found:", problem_numbers)
print("Averaging problems >=", CUTOFF_PROBLEM, ":", [p for p in problem_numbers if p >= CUTOFF_PROBLEM])


[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-27_date-20260124/behav/._MY_04_R-2026-01-24-124422.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-55_date-20260209/behav/._MY_04_R-2026-02-09-103918.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-28_date-20260124/behav/._MY_04_R-2026-01-24-165301.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-12_date-20260116/behav/._MY_04_R-2026-01-16-143640.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volum

## Helpers: per-problem curves + correct fraction-removed averaging

In [4]:
from src.behavior_visualization.plot_averaged_choice_probs import (
    compute_curve_from_windows,
    curves_by_problem_from_windows_dict,
    average_across_problem_curves,
    fraction_removed_by_problem,
    plot_choice_probs_with_fraction_removed_axis,
)

## Build per-problem reversal windows (and splits) for problems ≥ cutoff

In [5]:
windows_all = {}
windows_early_first2 = {}
windows_late_first2 = {}
windows_firsthalf = {}
windows_secondhalf = {}

windows_best2 = {}
windows_best3 = {}

for p in sorted(subjects_trials_by_problem.keys()):
    if p < CUTOFF_PROBLEM:
        continue

    subjects_trials = subjects_trials_by_problem[p]
    rw = get_good_reversal_info(subjects_trials, pre=pre, post=post, include_first_block=False)
    windows_all[p] = rw

    if DO_SPLIT_BY_FIRST_TWO:
        early2, late2 = split_good_reversals_early_late(rw, first_n=2)
        windows_early_first2[p] = early2
        windows_late_first2[p] = late2

    if DO_SPLIT_BY_HALF:
        firsthalf, secondhalf = split_good_reversals_early_late(rw)
        windows_firsthalf[p] = firsthalf
        windows_secondhalf[p] = secondhalf

    if DO_SPLIT_BY_BEST_CHANGE:
        b2, b3 = split_good_reversals_by_best_change(rw)
        windows_best2[p] = b2
        windows_best3[p] = b3

print("Built windows for problems:", sorted(windows_all.keys()))


Built windows for problems: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


## ALL averaged (with corrected fraction-removed cumulative axis)

In [6]:
if DO_ALL:
    curves_all = curves_by_problem_from_windows_dict(
        windows_all, pre=pre, post=post, skip_n=0, do_moving_avg=False, moving_avg_window=moving_avg_window
    )
    x_all_avg, across_all_avg = average_across_problem_curves(curves_all)

    frac_removed_all, probs_frac = fraction_removed_by_problem(windows_all, curves_all, x_all_avg, exclude_anchor=True)
    print("Fraction-removed computed from problems:", probs_frac)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Averaged)"
    plot_choice_probs_with_fraction_removed_axis(
        x_all_avg, across_all_avg, frac_removed_all, save_path=save_path
    )


Fraction-removed computed from problems: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:115: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:116: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:117: RuntimeWarning: Mean of empty slice
  "third_mean": np.nanmean(third_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_visualization/plot_averaged_choice_probs.py:223: RuntimeWarning: Mean of empty slice
  frac_avg = np.nanmean(F, axis=0)


## ALL averaged — Early / Late (first 2) with corrected fraction-removed axis

In [7]:
if DO_ALL and DO_SPLIT_BY_FIRST_TWO and windows_early_first2:
    # Early
    curves_early = curves_by_problem_from_windows_dict(
        windows_early_first2, pre=pre, post=post, skip_n=0, do_moving_avg=False, moving_avg_window=moving_avg_window
    )
    x_early, across_early = average_across_problem_curves(curves_early)
    frac_removed_early, _ = fraction_removed_by_problem(windows_early_first2, curves_early, x_early, exclude_anchor=True)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Averaged) (Early)"
    plot_choice_probs_with_fraction_removed_axis(x_early, across_early, frac_removed_early, save_path=save_path)

    # Late
    curves_late = curves_by_problem_from_windows_dict(
        windows_late_first2, pre=pre, post=post, skip_n=0, do_moving_avg=False, moving_avg_window=moving_avg_window
    )
    x_late, across_late = average_across_problem_curves(curves_late)
    frac_removed_late, _ = fraction_removed_by_problem(windows_late_first2, curves_late, x_late, exclude_anchor=True)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Averaged) (Late)"
    plot_choice_probs_with_fraction_removed_axis(x_late, across_late, frac_removed_late, save_path=save_path)


/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:115: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:116: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:117: RuntimeWarning: Mean of empty slice
  "third_mean": np.nanmean(third_mat, axis=0),
/Users/megyoung/anaconda3/envs/magnitude-bandit-analysis/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:135: RuntimeWarning: Mean of empty slice
  across_mean[k] = np.nanmea

[INFO] Skipping subject MY_04_R with no good reversals.


/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:115: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:116: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:117: RuntimeWarning: Mean of empty slice
  "third_mean": np.nanmean(third_mat, axis=0),
/Users/megyoung/anaconda3/envs/magnitude-bandit-analysis/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/megyoung/magnitude-bandit-analysis/src/behavior_visualization/plot_averaged_choice_probs.py:223: RuntimeWarning: Mean of empty slice
  frac_avg = np.nanmean(F, axis=0)


## SKIP family (averaged) — unchanged (no cumulative axis in your script)

In [8]:
if DO_SKIP:
    curves_skip = curves_by_problem_from_windows_dict(
        windows_all, pre=pre, post=post, skip_n=skip_n_trials_after_reversal, do_moving_avg=False, moving_avg_window=moving_avg_window
    )
    x_skip, across_skip = average_across_problem_curves(curves_skip)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals, Skipping Initial Trials (Averaged)"
    plot_choice_probs_around_good_reversals(
        x_skip, across_skip,
        add_cumulative_axis=False,
        skip_n_trials_after_reversal=skip_n_trials_after_reversal,
        save_path=save_path
    )

    if DO_SPLIT_BY_FIRST_TWO and windows_early_first2:
        curves_skip_early = curves_by_problem_from_windows_dict(
            windows_early_first2, pre=pre, post=post, skip_n=skip_n_trials_after_reversal, do_moving_avg=False, moving_avg_window=moving_avg_window
        )
        x_skip_early, across_skip_early = average_across_problem_curves(curves_skip_early)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals, Skipping Initial Trials (Averaged) (Early)"
        plot_choice_probs_around_good_reversals(
            x_skip_early, across_skip_early,
            add_cumulative_axis=False,
            skip_n_trials_after_reversal=skip_n_trials_after_reversal,
            save_path=save_path
        )

        curves_skip_late = curves_by_problem_from_windows_dict(
            windows_late_first2, pre=pre, post=post, skip_n=skip_n_trials_after_reversal, do_moving_avg=False, moving_avg_window=moving_avg_window
        )
        x_skip_late, across_skip_late = average_across_problem_curves(curves_skip_late)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals, Skipping Initial Trials (Averaged) (Late)"
        plot_choice_probs_around_good_reversals(
            x_skip_late, across_skip_late,
            add_cumulative_axis=False,
            skip_n_trials_after_reversal=skip_n_trials_after_reversal,
            save_path=save_path
        )


/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:115: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:116: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:117: RuntimeWarning: Mean of empty slice
  "third_mean": np.nanmean(third_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:115: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:116: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/ma

[INFO] Skipping subject MY_04_R with no good reversals.


/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:115: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:116: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:117: RuntimeWarning: Mean of empty slice
  "third_mean": np.nanmean(third_mat, axis=0),
/Users/megyoung/anaconda3/envs/magnitude-bandit-analysis/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


## MOVING AVERAGE family (averaged) with corrected fraction-removed axis

In [9]:
if DO_MOVING_AVG:
    curves_ma = curves_by_problem_from_windows_dict(
        windows_all, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
    )
    x_ma, across_ma = average_across_problem_curves(curves_ma)
    frac_removed_ma, _ = fraction_removed_by_problem(windows_all, curves_ma, x_ma, exclude_anchor=True)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Moving Average) (Averaged)"
    plot_choice_probs_with_fraction_removed_axis(x_ma, across_ma, frac_removed_ma, save_path=save_path)

    if DO_SPLIT_BY_HALF and windows_firsthalf:
        curves_ma_first = curves_by_problem_from_windows_dict(
            windows_firsthalf, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
        )
        x_ma_first, across_ma_first = average_across_problem_curves(curves_ma_first)
        frac_removed_first, _ = fraction_removed_by_problem(windows_firsthalf, curves_ma_first, x_ma_first, exclude_anchor=True)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Moving Average) (Averaged) (First Half)"
        plot_choice_probs_with_fraction_removed_axis(x_ma_first, across_ma_first, frac_removed_first, save_path=save_path)

        curves_ma_second = curves_by_problem_from_windows_dict(
            windows_secondhalf, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
        )
        x_ma_second, across_ma_second = average_across_problem_curves(curves_ma_second)
        frac_removed_second, _ = fraction_removed_by_problem(windows_secondhalf, curves_ma_second, x_ma_second, exclude_anchor=True)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Moving Average) (Averaged) (Second Half)"
        plot_choice_probs_with_fraction_removed_axis(x_ma_second, across_ma_second, frac_removed_second, save_path=save_path)

    if DO_SPLIT_BY_FIRST_TWO and windows_early_first2:
        curves_ma_early = curves_by_problem_from_windows_dict(
            windows_early_first2, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
        )
        x_ma_early, across_ma_early = average_across_problem_curves(curves_ma_early)
        frac_removed_ma_early, _ = fraction_removed_by_problem(windows_early_first2, curves_ma_early, x_ma_early, exclude_anchor=True)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Moving Average) (Averaged) (Early)"
        plot_choice_probs_with_fraction_removed_axis(x_ma_early, across_ma_early, frac_removed_ma_early, save_path=save_path)

        curves_ma_late = curves_by_problem_from_windows_dict(
            windows_late_first2, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
        )
        x_ma_late, across_ma_late = average_across_problem_curves(curves_ma_late)
        frac_removed_ma_late, _ = fraction_removed_by_problem(windows_late_first2, curves_ma_late, x_ma_late, exclude_anchor=True)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Moving Average) (Averaged) (Late)"
        plot_choice_probs_with_fraction_removed_axis(x_ma_late, across_ma_late, frac_removed_ma_late, save_path=save_path)


/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:115: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:116: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:117: RuntimeWarning: Mean of empty slice
  "third_mean": np.nanmean(third_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_visualization/plot_averaged_choice_probs.py:223: RuntimeWarning: Mean of empty slice
  frac_avg = np.nanmean(F, axis=0)
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:115: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysi

[INFO] Skipping subject MY_04_R with no good reversals.


/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:115: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:116: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:117: RuntimeWarning: Mean of empty slice
  "third_mean": np.nanmean(third_mat, axis=0),
/Users/megyoung/anaconda3/envs/magnitude-bandit-analysis/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/megyoung/magnitude-bandit-analysis/src/behavior_visualization/plot_averaged_choice_probs.py:223: RuntimeWarning: Mean of empty slice
  frac_avg = np.nanmean(F, axis=0)


## Optional: BEST-CHANGE (moving average) with corrected fraction-removed axis

In [10]:
if DO_SPLIT_BY_BEST_CHANGE and DO_MOVING_AVG:
    # best -> second
    curves_b2 = curves_by_problem_from_windows_dict(
        windows_best2, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
    )
    x_b2, across_b2 = average_across_problem_curves(curves_b2)
    frac_removed_b2, _ = fraction_removed_by_problem(windows_best2, curves_b2, x_b2, exclude_anchor=True)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probs Good Reversals (Best->Second) (Moving Average) (Averaged)"
    plot_choice_probs_with_fraction_removed_axis(x_b2, across_b2, frac_removed_b2, save_path=save_path)

    # best -> third
    curves_b3 = curves_by_problem_from_windows_dict(
        windows_best3, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
    )
    x_b3, across_b3 = average_across_problem_curves(curves_b3)
    frac_removed_b3, _ = fraction_removed_by_problem(windows_best3, curves_b3, x_b3, exclude_anchor=True)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probs Good Reversals (Best->Third) (Moving Average) (Averaged)"
    plot_choice_probs_with_fraction_removed_axis(x_b3, across_b3, frac_removed_b3, save_path=save_path)
